In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
from sklearn.preprocessing import MinMaxScaler

In [2]:
eref, Ce, Campl, Cp, CY, CN1, CN2, CN3, phic = 0.787, 0.45, 1.93, 0.11, 1.24, 4.85e-4, 0.093, 7.5e-5, 32.1

In [3]:
def HCA(N,deltaN,etaav,epsampl,e,pav,results):
  epsvacc,gA,gArate,fN,ffff = results
  fampl = (epsampl/0.0001)**Campl;
  fe = (Ce - e) * (Ce - e) / (1 + e) * (1 + eref) / (Ce - eref) / (Ce - eref);
  fp = np.exp(-Cp * ((pav)/100 -1.0));
  Yav = 27 * (3.0 + etaav)/(3.0 + 2.0 * etaav) / (3.0 - etaav);
  Yc = (9.0 - np.sin(phic * 3.141592654/180.0) * np.sin(phic * 3.141592654/180.0))\
  / (1.0 - np.sin(phic * 3.141592654/180.0) * np.sin(phic * 3.141592654/180.0));
  Yquerav = (Yav - 9.0) / (Yc - 9.0);
  fY = np.exp(CY * Yquerav);
  fNArate = CN1 * CN2 * np.exp(-gA/(CN1 * fampl));
  fNBrate = CN1 * CN3;
  fNrate = fNArate + fNBrate;
  epsvaccrate = np.sqrt(3.0) *  fampl * fe * fp * fY * fNrate;
  epsvacc = epsvacc + epsvaccrate *deltaN;
  fN = fN + fNrate*deltaN
  gArate = fampl * fNArate
  gA = gA + gArate*deltaN
  ffff = ffff+fampl*fe*fp*fY*fNrate*deltaN
  return [epsvacc,gA,gArate,fN,ffff]

In [4]:
rng1 = np.random.default_rng(seed=1234)
n_simulations, timesteps = 100_000, 16
n_out_var = 2 + timesteps
X = np.zeros((n_simulations, 5))
Y = np.zeros((n_simulations, n_out_var))
for j in range(n_simulations):
  etaav = rng1.uniform(-0.6, 1.0)
  pav = rng1.uniform(50.0, 800.0)
  e0 = rng1.uniform(0.521, eref)
  epsampl = rng1.uniform(1.0e-5, 7.0e-4)
  Nmax = int(np.round(rng1.uniform(1e4, 1e6)))
  X[j] = [etaav, pav, e0, epsampl, Nmax]
  N_steps = np.logspace(start=0, stop=np.log10(Nmax), num=timesteps, base=10.0, dtype=int)
  epsacc_list, ffffN = [], []
  state = [0.0, 0.0, 0.0, 0.0, 0.0]
  for i in range(len(N_steps)):
    if(i>0):
      state = HCA(N_steps[i], N_steps[i]-N_steps[i-1], etaav, epsampl, e0, pav, state)
      epsacc_list.append(-state[0])
      ffffN.append(state[-1])
    else:
      state = HCA(N_steps[i], 1, etaav, epsampl, e0, pav, state)
      epsacc_list.append(-state[0])
      ffffN.append(state[-1])
  Y[j] = [etaav, e0] + ffffN

In [5]:
scaler_target_min, scaler_target_max = -1.0, 1.0
scaler_input = MinMaxScaler(feature_range=(scaler_target_min, scaler_target_max))
Xscaled = scaler_input.fit_transform(X)
total_size = n_simulations
train_size, test_size = round(total_size*0.6), round(total_size*0.2)
learning_size = train_size + test_size
validation_size = total_size - learning_size
input_train = Xscaled[0:train_size]
input_test = Xscaled[train_size:learning_size]
input_validation = Xscaled[learning_size:total_size]
Y[:, 2:] = np.clip(Y[:, 2:], None, 1.2)
output_train = Y[0:train_size]
output_test = Y[train_size:learning_size]
output_validation = Y[learning_size:total_size]

In [6]:
eps = 1e-8
def tailored_loss(y_true, y_pred):
  epsacc_true, epsacc_pred = y_true[..., 2:], y_pred[..., 2:]
  DD_error = tf.reduce_mean(tf.square(epsacc_true - epsacc_pred))
  epsacc = y_pred[:, 2:]
  epsacc_rate = epsacc[:, 1:] - epsacc[:, :-1]
  PI_error1 = tf.reduce_sum(tf.cast(epsacc_rate < 0.0, tf.float32) * 2.0)
  true_etaav, true_e0 = y_true[:, 0], y_true[:, 1]
  phi_c = phic * tf.constant(np.pi / 180.0, dtype=tf.float32)
  M = 6 * tf.sin(phi_c) / (3 - tf.sin(phi_c))
  f = 1 / tf.sqrt((1/3) * (1 - (true_etaav**2)/(M**2))**2 + 6 * (true_etaav/(M**2))**2)
  mv = f * (1 - (true_etaav**2) / (M**2))
  mv = tf.expand_dims(mv, axis=1)
  epsv_true = epsacc_true * mv
  epsv_pred = epsacc_pred * mv
  n_steps = epsv_true.shape[1]
  e_true_list, e_pred_list = [], []
  e_prev_true, e_prev_pred = true_e0, true_e0
  for n in range(n_steps):
    if n == 0:
      depsv_true_n = epsv_true[:, 0]
      depsv_pred_n = epsv_pred[:, 0]
    else:
      depsv_true_n = epsv_true[:, n] - epsv_true[:, n - 1]
      depsv_pred_n = epsv_pred[:, n] - epsv_pred[:, n - 1]
    e_now_true = e_prev_true + (1.0 + e_prev_true) * depsv_true_n
    e_now_pred = e_prev_pred + (1.0 + e_prev_pred) * depsv_pred_n
    e_true_list.append(e_now_true)
    e_pred_list.append(e_now_pred)
    e_prev_true = e_now_true
    e_prev_pred = e_now_pred
  e_true = tf.stack(e_true_list, axis=1)
  e_pred = tf.stack(e_pred_list, axis=1)
  PI_error2 = tf.reduce_mean(tf.square(e_true - e_pred))
  losses = tf.stack([DD_error, PI_error1, PI_error2])
  orders = tf.floor(tf.math.log(losses + eps) / tf.math.log(tf.constant(10.0, dtype=losses.dtype)))
  scales = tf.pow(10.0, orders)
  scales = tf.stop_gradient(scales)
  scaled_losses = losses / scales
  total_loss = tf.reduce_sum(scaled_losses)
  return total_loss

In [7]:
h = int(280)
def build_model():
  model = keras.Sequential([
    layers.Input(shape=[5]),
    layers.Dense(h*1, activation="tanh"),
    layers.Dense(h*2, activation="gelu"),
    layers.Dense(h*4, activation="gelu"),
    layers.Dense(h*4, activation="gelu"),
    layers.Dense(h*2, activation="gelu"),
    layers.Dense(h*1, activation="gelu"),
    layers.Dense(n_out_var, activation="sigmoid")])
  optimizer = optimizers.Adamax(learning_rate=0.0002, beta_1=0.9, beta_2=0.999, epsilon=eps, name="Adamax")
  model.compile(loss=tailored_loss, optimizer=optimizer)
  return model
model = build_model()
history = model.fit(input_train, output_train, validation_data=(input_test, output_test), batch_size=32, epochs=30, verbose=0)
pred_train, pred_test, pred_validation = model.predict(input_train), model.predict(input_test), model.predict(input_validation)
model.summary()

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 280)            │         1,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 560)            │       157,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1120)           │       628,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1120)           │     1,255,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 560)            │       627,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 280)            │       157,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 18)             │         5,058 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,498,336 (32.42 MB)

 Trainable params: 2,832,778 (10.81 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,665,558 (21.61 MB)